In [1]:

import gc
import json
import sys
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing.pipeline import (
    stratified_split,
    fit_scaler,
    apply_scaler,
    smote_train_only,
    remove_invalid_negative_rows,
)

LABEL = "Attack Type"
NORMAL = "Normal Traffic"
SEED = 42

df = pd.read_parquet(
    ROOT / "data" / "interim" / "cleaned.parquet"
)

df, removed = remove_invalid_negative_rows(df)

print("Invalid negative rows removed:", removed)
print("Remaining records:", len(df))

assert removed == 2909
assert len(df) == 2517681
assert df["Attack Type"].nunique() == 7

schema = json.loads(
    (ROOT / "data" / "schema" / "feature_schema.json").read_text(
        encoding="utf-8"
    )
)

FEATURE_COLUMNS = [
    feature["name"]
    for feature in schema["features"]
    if feature["name"] not in {
        "Active Mean",
        "Active Max",
        "Active Min",
        "Idle Mean",
        "Idle Max",
        "Idle Min",
    }
]

DROP_FEATURES = {
    "Active Mean",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Max",
    "Idle Min",
}
assert len(FEATURE_COLUMNS) == 46
assert "Destination Port" in FEATURE_COLUMNS
assert DROP_FEATURES.isdisjoint(FEATURE_COLUMNS)

X = df[FEATURE_COLUMNS].copy()
y = df[LABEL]

print("Selected features:", X.shape[1])
print("Dataset shape:", X.shape)
print("Destination Port retained:", "Destination Port" in X.columns)
print(X.shape, y.shape)

del df
gc.collect()


Invalid negative rows removed: 2909
Remaining records: 2517681
Selected features: 46
Dataset shape: (2517681, 46)
Destination Port retained: True
(2517681, 46) (2517681,)


0

In [2]:

le = joblib.load(ROOT / "models" / "label_encoder.pkl")

y_enc = pd.Series(
    le.transform(y),
    index=y.index,
    name="label"
)

mapping = {
    int(i): str(name)
    for i, name in enumerate(le.classes_)
}

print(mapping)

(ROOT / "models").mkdir(exist_ok=True)

import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing.pipeline import (
    stratified_split,
    fit_scaler,
    apply_scaler,
    smote_train_only,
)



{0: 'Bots', 1: 'Brute Force', 2: 'DDoS', 3: 'DoS', 4: 'Normal Traffic', 5: 'Port Scanning', 6: 'Web Attacks'}


In [3]:

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(
    X, y_enc, seed=SEED
)

for name, part in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test),
]:
    share = part.value_counts(normalize=True).sort_index().round(4)
    print(name, len(part), dict(share))

assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

print("Split checks passed")


for name, part in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test)
]:
    share = part.value_counts(normalize=True).sort_index().round(4)
    print(name, len(part), dict(share))

assert not (set(X_train.index) & set(X_val.index))
assert not (set(X_train.index) & set(X_test.index))
assert not (set(X_val.index) & set(X_test.index))

print("Split checks passed")


train 1762376 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.831), 5: np.float64(0.036), 6: np.float64(0.0009)}
val 377652 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.831), 5: np.float64(0.036), 6: np.float64(0.0009)}
test 377653 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.831), 5: np.float64(0.036), 6: np.float64(0.0008)}
Split checks passed
train 1762376 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.831), 5: np.float64(0.036), 6: np.float64(0.0009)}
val 377652 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64(0.831), 5: np.float64(0.036), 6: np.float64(0.0009)}
test 377653 {0: np.float64(0.0008), 1: np.float64(0.0036), 2: np.float64(0.0508), 3: np.float64(0.0769), 4: np.float64

In [4]:

INTERIM = ROOT / "data" / "interim"

for name, Xp, yp in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test)
]:
    out = Xp.copy()
    out["label"] = yp
    print(f"Verified split_{name}_unscaled: {out.shape}")
    del out

print("Split shapes verified without saving files")


Verified split_train_unscaled: (1762376, 47)
Verified split_val_unscaled: (377652, 47)
Verified split_test_unscaled: (377653, 47)
Split shapes verified without saving files


In [5]:

schema = json.loads(
    (ROOT / "data" / "schema" / "feature_schema.json").read_text(
        encoding="utf-8"
    )
)


schema_cols = [
    f["name"]
    for f in schema["features"]
    if f["name"] not in DROP_FEATURES
]

assert len(schema_cols) == 46
assert list(X_train.columns) == schema_cols, (
    "Columns differ from selected 46 features"
)

print("Feature schema validation passed:", len(schema_cols))


print("Columns match feature_schema.json")
print("Feature count:", len(schema_cols))


Feature schema validation passed: 46
Columns match feature_schema.json
Feature count: 46


In [6]:

scaler = joblib.load(ROOT / "models" / "scaler.pkl")

X_train_s = apply_scaler(scaler, X_train)
X_val_s = apply_scaler(scaler, X_val)
X_test_s = apply_scaler(scaler, X_test)

scaler_metadata = {
    "feature_names": scaler.feature_names_in_.tolist(),
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "variance": scaler.var_.tolist(),
    "n_features": int(scaler.n_features_in_),
    "n_samples_seen": int(scaler.n_samples_seen_),
}

print(
    "Largest train column average (should be about 0):",
    X_train_s.mean().abs().max()
)

print("Existing scaler reused successfully")

del X_train, X_val, X_test
gc.collect()

Largest train column average (should be about 0): 1.1682346055203124e-16
Existing scaler reused successfully


20

In [7]:

train_counts = Counter(y_train)

print("Train class counts:", dict(sorted(train_counts.items())))

FLOOR = 10000

strategy = {
    cls: FLOOR
    for cls, count in train_counts.items()
    if count < FLOOR
}

print("Classes raised to", FLOOR, ":", strategy)

X_res, y_res = smote_train_only(
    X_train_s,
    y_train,
    floor=FLOOR,
    seed=SEED
)

print("After SMOTE:", dict(sorted(Counter(y_res).items())))

print(
    "Train rows",
    len(X_train_s),
    "->",
    len(X_res),
    "| val and test untouched:",
    len(X_val_s),
    len(X_test_s)
)


Train class counts: {0: 1364, 1: 6401, 2: 89596, 3: 135507, 4: 1464522, 5: 63486, 6: 1500}
Classes raised to 10000 : {1: 10000, 0: 10000, 6: 10000}
After SMOTE: {0: 10000, 1: 10000, 2: 89596, 3: 135507, 4: 1464522, 5: 63486, 6: 10000}
Train rows 1762376 -> 1783111 | val and test untouched: 377652 377653


In [8]:

import pyarrow.parquet as pq

path = ROOT / "data/processed/train_smote_floor10000.parquet"

labels = pq.read_table(path, columns=["label"]).column("label").to_numpy()

print("Total SMOTE rows:", len(labels))
print("Class distribution:", dict(sorted(Counter(labels).items())))
print("Minority floor satisfied:", all(
    count >= 10000 for count in Counter(labels).values()
))


Total SMOTE rows: 1783111
Class distribution: {np.int64(0): 10000, np.int64(1): 10000, np.int64(2): 89596, np.int64(3): 135507, np.int64(4): 1464522, np.int64(5): 63486, np.int64(6): 10000}
Minority floor satisfied: True


In [9]:

PROC = ROOT / "data" / "processed"

expected = {
    "train_scaled.parquet": 1762376,
    "train_smote_floor10000.parquet": 1783111,
    "val_scaled.parquet": 377652,
    "test_scaled.parquet": 377653,
}

for filename, expected_rows in expected.items():
    path = PROC / filename

    if not path.exists():
        print(f"MISSING: {filename}")
        continue

    import pyarrow.parquet as pq
    metadata = pq.read_metadata(path)

    print(
        filename,
        "Rows:", metadata.num_rows,
        "Columns:", metadata.num_columns,
        "Match:", metadata.num_rows == expected_rows
        and metadata.num_columns == 47
    )

print("Processed dataset verification complete")


train_scaled.parquet Rows: 1762376 Columns: 47 Match: True
train_smote_floor10000.parquet Rows: 1783111 Columns: 47 Match: True
val_scaled.parquet Rows: 377652 Columns: 47 Match: True
test_scaled.parquet Rows: 377653 Columns: 47 Match: True
Processed dataset verification complete
